# RadarML Lite Track 2 — compact complex-STFT neural classifier

This notebook trains a **neural network from random initialization**, with gradient
descent and backpropagation. Its inference is **one network forward pass** over an
explicit time–frequency representation. It contains **no waveform-template bank,
nearest-neighbor lookup, exhaustive configuration search or previous-prediction input**.

The architecture combines learned local complex-STFT features, a small global
phase-sensitive projection and a lightweight CNN over STFT power. Preserving complex
information addresses a limitation of the original power-only spectrogram CNN.
There are approximately **2.48 million trainable parameters / 10 MB of FP32 weights**.
This is a new experiment: **0.996 is a target, not an established accuracy claim**.
The earlier template-bank score does not transfer to this network.

**Kaggle:** attach the same expanded `radar-ml` dataset **with JSON sidecars**, the
published `manifests` dataset and the released `blind-test/participants` dataset.
Enable a GPU and **Run All**. Data/manifest/participants paths are auto-detected;
manual examples matching your previous notebook are in the next cell. The smaller
compact `recorded_dataset` is also supported. No Internet or extra code upload is needed.

The default mode trains, selects a checkpoint, evaluates all **171,600 validation
examples**, exports the model and inference script, and generates a submission on
**CPU**, including measured STFT time. The training budget is **10.5 hours**, leaving
room within your 12-hour session for validation/export/inference. Actual duration
and performance depend on hardware and I/O; the code cannot enforce Kaggle's session
lifetime. It saves epoch checkpoints so an interrupted run can continue.

The organizer's model-freeze restriction was removed according to your update.
Inference is designed for the newer restriction: **model weights + blind signals only**.
Architecture and measured runtime are disclosed; the organizer determines acceptance.

In [ ]:
from pathlib import Path
import copy
import importlib
import json
import shutil
import subprocess
import sys

MODE = "train_and_submit"       # "train_and_submit", "train", "infer"
RUN_NAME = "track2_neural_v1"    # use a new name for a new architecture/configuration
DATA_FORMAT = "auto"            # "auto", "expanded", "compact"
DATA_ROOT = None
MANIFEST_PATH = None
BLIND_ROOT = None
# Known paths from your previous run, if you prefer to set them explicitly:
# DATA_ROOT = Path("/kaggle/input/datasets/hadimanes/radar-ml/expanded (2)/expanded")
# MANIFEST_PATH = Path("/kaggle/input/datasets/hadimanes/manifests")
# BLIND_ROOT = Path("/kaggle/input/datasets/hadimanes/blind-test/participants")

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
RUN_DIR = WORK_ROOT / RUN_NAME
RESUME_FROM = None              # extracted *_resume.zip folder from a saved session
MODEL_PATH = None               # only for MODE="infer": exported neural model.pt

TRAIN_BATCH_SIZE = 256          # reduce to 128 if GPU memory is insufficient
WORKERS = 2                    # use 0 if notebook multiprocessing causes trouble
TRAINING_BUDGET_HOURS = 10.5
REMOVE_TRAINING_CACHE_AFTER_EXPORT = True

INFERENCE_DEVICE = "cpu"        # CPU by default for the organizer's timing requirement
INFERENCE_BATCH_SIZE = 128
INFERENCE_CPU_THREADS = 4
BENCHMARK_SAMPLES = 1000         # real one-signal-at-a-time latency, separate from batching
SUBMISSION_LABEL_COLUMN = "label"  # matches your latest submission; use "predicted_label" if required
PREDICTIONS_PATH = WORK_ROOT / "submission_track2_neural.csv"

assert MODE in {"train_and_submit", "train", "infer"}
assert Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {"", ".", ".."}
TRAIN_MODEL = MODE in {"train", "train_and_submit"}
GENERATE_SUBMISSION = MODE in {"train_and_submit", "infer"}
if TRAIN_MODEL and RESUME_FROM is not None:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("resume.pt", "best.pt", "run_config.json", "history.json", "best_selection_validation.json"):
        source = Path(RESUME_FROM) / name
        destination = RUN_DIR / name
        if source.exists() and not destination.exists():
            shutil.copy2(source, destination)
if not TRAIN_MODEL:
    assert MODEL_PATH is not None, "Set MODEL_PATH to the new neural model.pt for inference-only mode"
print("Mode:", MODE, "| run:", RUN_DIR)


## Embedded code

All source is included. The first module retains the original validated file-identity
and split helpers, including FSK filename-collision recovery. It is used for development
only. The second module is the **standalone inference script** and model definition.
The third module handles training and is excluded from the inference export.

The exported inference script does not import the development modules, open training
files, require the training cache, or read an earlier submission. Its inputs are just
`model.pt` and the anonymous blind package (signals, index and submission template).

In [ ]:
%%writefile radarml_track2_runtime.py
"""RadarML Lite Track 2 / Track B: from-scratch STFT classification.
SPDX-License-Identifier: GPL-3.0-or-later
Lookup code adapted from the participant's synapseon.ipynb and the organizer's
GPL-3.0 utilities. RadarML attribution: Ritchie, White and Hosford (2025).
AI coding assistance: OpenAI Codex. No pretrained weights or external data.
"""
import argparse
import csv
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

CLASSES = ("barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad")
ALL_SNRS = (30, 25, 20, 15, 10, 5, 0, -5, -10, -15, -20, -25, -30)
SAMPLE_RATE_HZ = 120_000_000
MANIFEST_SHA256 = "4ccc4937f4ba0e9993837bb26a5ba4463f1a880b689a753ad3a0e39b76ebafc7"

# %% File identity and published split
def family(filename):
    parts = Path(filename).stem.lower().split("_")
    if len(parts) < 3:
        raise ValueError(f"Unrecognized waveform filename: {filename}")
    token = parts[1]
    if token.startswith("barker"):
        return "barker"
    if re.fullmatch(r"fsk(?:[#-]?[0-3])?", token):
        return "fsk"
    if token == "fsk" and len(parts) > 2 and parts[2] in {"0", "1", "2", "3"}:
        return "fsk"
    if token in CLASSES:
        return token
    raise ValueError(f"Unrecognized modulation in {filename}: {token}")

def canonical_filename(filename):
    # Kaggle can append " (19)" before an extension. Preserve frequency signs.
    filename = re.sub(r" \(\d+\)(?=\.(?:npy|json)$)", "", filename)
    return re.sub(r"(?<=_)fsk[#_-]?([0-3])(?=_)", r"fsk\1", filename, flags=re.IGNORECASE)

def expanded_name_from_source(source_file):
    """Recreate the organizer's expanded name from JSON's original source_file."""
    source = Path(source_file).name
    if source.endswith("_recorded.npy"):
        return source.replace("_recorded.npy", "_single_channel.npy")
    if source.endswith("_single_channel.npy"):
        return source
    parts = Path(source).stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected source_file in JSON sidecar: {source}")
    try:
        parts[-1] = "-".join(f'{240 - int(piece.removesuffix("Mhz"))}Mhz'
                             for piece in parts[-1].split("-"))
    except ValueError as exc:
        raise ValueError(f"Unexpected source frequency in JSON sidecar: {source}") from exc
    return "_".join(parts) + "_single_channel.npy"

def build_file_lookup(root, kind):
    suffix = "_single_channel.npy" if kind == "expanded" else "_recorded.npy"
    files = [p for p in root.rglob("*.npy") if canonical_filename(p.name).endswith(suffix)]
    if not files:
        raise FileNotFoundError(f"No {suffix} files in {root}")
    file_set = set(files)
    arrays, metadata = defaultdict(set), defaultdict(list)
    for path in files:
        arrays[canonical_filename(path.name)].add(path)
    for sidecar in root.rglob("*.json"):
        meta = json.loads(sidecar.read_text(encoding="utf-8"))
        if not isinstance(meta, dict) or not meta.get("source_file"):
            continue
        alias = expanded_name_from_source(meta["source_file"])
        if kind == "compact":
            alias = alias.replace("_single_channel.npy", "_recorded.npy")
        key = canonical_filename(alias)
        metadata[key].append((sidecar, meta))
        paired_array = sidecar.with_suffix(".npy")
        if paired_array in file_set:
            arrays[key].add(paired_array)
    hash_to_paths, path_to_hash = defaultdict(list), {}
    if kind == "expanded":
        fsk_paths = [p for p in files if family(canonical_filename(p.name)) == "fsk"]
        print(f"Hashing {len(fsk_paths)} uploaded FSK arrays to recover their original identities...")
        for count, path in enumerate(fsk_paths, 1):
            digest = lookup_sha256(path)
            hash_to_paths[digest].append(path)
            path_to_hash[path] = digest
            if count % 25 == 0 or count == len(fsk_paths):
                print(f"Hashed {count}/{len(fsk_paths)} FSK arrays")
        for key, entries in metadata.items():
            if "_fsk" in key.lower():
                for _, meta in entries:
                    for path in hash_to_paths.get(meta.get("output_sha256"), []):
                        arrays[key].add(path)
    return {"arrays": arrays, "metadata": metadata,
            "hash_to_paths": hash_to_paths, "path_to_hash": path_to_hash}

def lookup_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_file(manifest_name, lookup, kind):
    desired = (manifest_name.replace("_recorded.npy", "_single_channel.npy")
               if kind == "expanded" else manifest_name)
    key = canonical_filename(desired)
    if kind == "expanded" and family(manifest_name) == "fsk":
        expected = {meta.get("output_sha256") for _, meta in lookup["metadata"].get(key, [])}
        digests_found = {digest for digest in expected if digest in lookup["hash_to_paths"]}
        if len(digests_found) == 1:
            digest = next(iter(digests_found))
            return sorted(lookup["hash_to_paths"][digest])[0]
        raise ValueError(f"FSK array for {manifest_name} cannot be matched by SHA256. "
                         f"Matching hashes found: {len(digests_found)}. "
                         "The Kaggle dataset may be missing an NPY file or its JSON may be stale.")
    matches = list(lookup["arrays"].get(key, []))
    paired = {sidecar.with_suffix(".npy") for sidecar, _ in lookup["metadata"].get(key, [])}
    if paired:
        matches = [path for path in matches if path in paired]
    if matches:
        shallowest = min(len(path.parts) for path in matches)
        matches = [path for path in matches if len(path.parts) == shallowest]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        expected_hashes = {meta.get("output_sha256") for _, meta in lookup["metadata"].get(key, [])}
        matching = [(path, digest) for path in matches
                    if (digest := lookup_sha256(path)) in expected_hashes]
        if matching and len({digest for _, digest in matching}) == 1:
            return sorted(path for path, _ in matching)[0]
    examples = [p.name for paths in lookup["arrays"].values() for p in paths
                if "fsk" in p.name.lower()][:5]
    raise ValueError(f"Could not uniquely match {manifest_name}: {len(matches)} candidate arrays. "
                     f"Example Kaggle FSK names: {examples}. Check array uploads and JSON checksums.")

def resolve_metadata(manifest_name, path, lookup, kind):
    desired = (manifest_name.replace("_recorded.npy", "_single_channel.npy")
               if kind == "expanded" else manifest_name)
    candidates = lookup["metadata"].get(canonical_filename(desired), [])
    if kind == "expanded" and family(manifest_name) == "fsk":
        digest = lookup["path_to_hash"].get(path)
        matched = [(sidecar, meta) for sidecar, meta in candidates
                   if meta.get("output_sha256") == digest]
        if matched:
            return sorted(matched, key=lambda pair: str(pair[0]))[0]
        raise ValueError(f"FSK metadata checksum missing for {manifest_name}")
    exact = [(sidecar, meta) for sidecar, meta in candidates if sidecar == path.with_suffix(".json")]
    if len(exact) == 1:
        return exact[0]
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        digest = lookup_sha256(path)
        matched = [(sidecar, meta) for sidecar, meta in candidates
                   if meta.get("output_sha256") == digest]
        if matched:
            return sorted(matched, key=lambda pair: str(pair[0]))[0]
    raise ValueError(f"Could not pair {path.name} with a JSON sidecar for {manifest_name}")

def load_manifest(path):
    path = Path(path)
    if lookup_sha256(path) != MANIFEST_SHA256:
        raise ValueError("Manifest checksum differs from the published release. Use the original CSV, not a regenerated split.")
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["file", "pulse_index", "split"]:
            raise ValueError("Expected manifest columns: file,pulse_index,split")
        rows = list(reader)
    if not rows:
        raise ValueError("Empty manifest")
    identities = set()
    per_file = Counter()
    for row in rows:
        if row["split"] not in {"train", "validation"}:
            raise ValueError(f"Unexpected split {row['split']}")
        pulse = int(row["pulse_index"])
        if not 0 <= pulse < 100:
            raise ValueError(f"Pulse index outside 0..99: {pulse}")
        identity = (row["file"], pulse)
        if identity in identities:
            raise ValueError(f"Duplicate source pulse: {identity}")
        identities.add(identity)
        per_file[(row["file"], row["split"])] += 1
        row["pulse_index"] = pulse
    for name in {r["file"] for r in rows}:
        if per_file[(name, "train")] != 80 or per_file[(name, "validation")] != 20:
            raise ValueError(f"Manifest is not 80/20 for {name}")
    if len(rows) != 66_000 or len(per_file) != 1_320:
        raise ValueError("Expected the complete published 660-recording, 66,000-pulse manifest")
    print(f"Manifest: {len(rows):,} source pulses across {len(per_file)//2} recordings")
    return rows


# %% Expanded data: no expansion or synthetic-noise generation
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)


def find_manifest(root, explicit=None):
    if explicit:
        path = Path(explicit)
        if path.is_dir():
            candidates = sorted(path.rglob("split_manifest.csv"))
        else:
            candidates = [path] if path.is_file() else []
    else:
        candidates = [p for p in (Path(root) / "split_manifest.csv",
                                  Path(root).parent / "split_manifest.csv") if p.is_file()]
        if not candidates and Path("/kaggle/input").is_dir():
            candidates = sorted(Path("/kaggle/input").rglob("split_manifest.csv"))
    # Multiple copies are fine only when all have the published identity.
    valid = [p for p in candidates if lookup_sha256(p) == MANIFEST_SHA256]
    if not valid:
        raise FileNotFoundError("Attach the published split_manifest.csv (repository hackathon/ copy); set MANIFEST_PATH to it.")
    return valid[0]


def prepare_development(root, manifest_path):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Expanded DATA_ROOT does not exist: {root}. Update the Kaggle path in the settings cell.")
    rows = load_manifest(manifest_path)
    lookup = build_file_lookup(root, "expanded")
    infos, paths_used = {}, set()
    for name in sorted({r["file"] for r in rows}):
        path = resolve_file(name, lookup, "expanded")
        sidecar, meta = resolve_metadata(name, path, lookup, "expanded")
        # Verify that the source identity really maps to this published recording.
        expected_name = name.replace("_recorded.npy", "_single_channel.npy")
        if canonical_filename(expanded_name_from_source(meta["source_file"])) != canonical_filename(expected_name):
            raise ValueError(f"Wrong source identity in {sidecar}")
        snrs = tuple(meta["snr_db"])
        if len(snrs) != 13 or set(snrs) != set(ALL_SNRS):
            raise ValueError(f"Expected all 13 requested SNRs in {sidecar}; found {snrs}")
        if meta.get("source_pulse_indices") != list(range(100)):
            raise ValueError(f"Pulse-axis mapping is not the organizer's 0..99 mapping: {sidecar}")
        if meta.get("sample_rate_hz") != SAMPLE_RATE_HZ or meta.get("channel") != "ADC0":
            raise ValueError(f"Unexpected receiver/sample rate: {sidecar}")
        array = np.load(path, mmap_mode="r", allow_pickle=False)
        if array.shape != (1, 13, 100, 33600) or array.dtype != np.int16:
            raise ValueError(f"Expected expanded int16 (1,13,100,33600): {path}, {array.shape}, {array.dtype}")
        if meta.get("output_shape") != list(array.shape) or meta.get("dtype") != "int16":
            raise ValueError(f"NPY/JSON layout disagreement: {path}")
        del array
        if path in paths_used:
            raise ValueError(f"Two source identities resolve to the same NPY: {path}")
        paths_used.add(path)
        infos[name] = {"path": str(path), "label": CLASSES.index(family(name)),
                       "snr_to_axis": {int(s): i for i, s in enumerate(snrs)}}
    print(f"Validated {len(infos)} expanded arrays; source pulses retain their published split.", flush=True)
    return rows, infos


def build_examples(rows, infos, split, pulse_limit, snrs):
    if split not in {"train", "validation"}:
        raise ValueError("Only the published train and validation splits are used here")
    limit = 80 if split == "train" else 20
    if not 1 <= pulse_limit <= limit or not snrs or not set(snrs).issubset(ALL_SNRS):
        raise ValueError("Invalid source-pulse limit or requested SNR subset")
    grouped = defaultdict(list)
    for row in rows:
        if row["split"] == split:
            grouped[row["file"]].append(row["pulse_index"])
    # SNR first improves sequential validation reads; selection uses original indices.
    return [(name, pulse, int(snr), infos[name]["label"])
            for name in sorted(grouped) for snr in snrs
            for pulse in sorted(grouped[name])[:pulse_limit]]


def interleaved_to_channels(row):
    if row.shape != (33600,) or row.dtype != np.int16:
        raise ValueError(f"Expected one interleaved int16 pulse: {row.shape}, {row.dtype}")
    return np.stack((row[0::2], row[1::2])).astype(np.float32) / 32768.0


class DevelopmentDataset(Dataset):
    def __init__(self, examples, infos):
        self.examples, self.infos = examples, infos

    @staticmethod
    @lru_cache(maxsize=24)
    def _array(path):
        return np.load(path, mmap_mode="r", allow_pickle=False)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        name, pulse, snr, label = self.examples[index]
        info = self.infos[name]
        row = self._array(info["path"])[0, info["snr_to_axis"][snr], pulse]
        # Label and requested SNR are returned for supervision/metrics only.
        return torch.from_numpy(interleaved_to_channels(row)), label, snr


def make_loader(dataset, batch_size, workers, seed, sampler=None):
    options = dict(batch_size=batch_size, num_workers=workers,
                   pin_memory=torch.cuda.is_available(), shuffle=False,
                   generator=torch.Generator().manual_seed(seed), worker_init_fn=seed_worker)
    if workers:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(dataset, sampler=sampler, **options)


# %% Track B representation and model, initialized from scratch
class TwoResolutionSTFT(nn.Module):
    def __init__(self, n_ffts=(256, 1024), hop_length=64, image_size=(256, 256)):
        super().__init__()
        self.n_ffts, self.hop_length, self.image_size = tuple(n_ffts), hop_length, tuple(image_size)
        for n_fft in self.n_ffts:
            self.register_buffer(f"window_{n_fft}", torch.hann_window(n_fft), persistent=False)

    def forward(self, x):
        # FFT remains float32/complex64 even during mixed-precision training.
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            rms = x.square().sum(1).mean(1, keepdim=True).sqrt().clamp_min(1e-6)
            iq = torch.complex(x[:, 0], x[:, 1]) / rms
            views = []
            for n_fft in self.n_ffts:
                window = getattr(self, f"window_{n_fft}")
                z = torch.stft(iq, n_fft=n_fft, hop_length=self.hop_length,
                               win_length=n_fft, window=window, center=True,
                               pad_mode="constant", onesided=False, return_complex=True)
                power = torch.fft.fftshift(z.abs().square(), dim=1) / window.square().sum()
                # Fixed acquisition window, never filename pulse duration or SNR label.
                # Use only frames fully inside the 0..20 us noise interval.
                first = math.ceil((n_fft / 2) / self.hop_length)
                stop = math.floor((2400 - n_fft / 2) / self.hop_length) + 1
                noise_floor = power[:, :, first:stop].mean((1, 2), keepdim=True).clamp_min(1e-8)
                view = torch.log1p(power / noise_floor).unsqueeze(1)
                # Averaging preserves narrow-band energy without taking only the largest bin.
                views.append(F.adaptive_avg_pool2d(view, self.image_size))
            return torch.cat(views, dim=1)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
            nn.GroupNorm(8, out_channels), nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.GroupNorm(8, out_channels))
        self.skip = (nn.Identity() if in_channels == out_channels and stride == 1 else
                     nn.Conv2d(in_channels, out_channels, 1, stride, bias=False))

    def forward(self, x):
        return F.silu(self.main(x) + self.skip(x))


class Track2Model(nn.Module):
    def __init__(self, n_ffts=(256, 1024), hop_length=64, image_size=(256, 256), dropout=0.2):
        super().__init__()
        self.representation = TwoResolutionSTFT(n_ffts, hop_length, image_size)
        self.features = nn.Sequential(
            nn.Conv2d(len(n_ffts), 24, 5, 2, 2, bias=False),
            nn.GroupNorm(8, 24), nn.SiLU(inplace=True),
            ResidualBlock(24, 48, 2), ResidualBlock(48, 48),
            ResidualBlock(48, 96, 2), ResidualBlock(96, 96),
            ResidualBlock(96, 128, 2), ResidualBlock(128, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(128 * 4 * 4, 256),
                                  nn.SiLU(), nn.Dropout(dropout), nn.Linear(256, len(CLASSES)))

    def forward(self, iq):
        return self.head(self.features(self.representation(iq)))


# %% Organizer-compatible metric and measured evaluation
def report_from_matrices(matrices):
    per_snr = []
    for snr in ALL_SNRS:
        if snr not in matrices:
            continue
        matrix = matrices[snr]
        totals = matrix.sum(1)
        if (totals == 0).any():
            raise ValueError(f"Requested SNR {snr} lacks one or more classes")
        per_snr.append({"requested_snr_db": int(snr),
                        "balanced_accuracy": float(np.mean(matrix.diagonal() / totals)),
                        "accuracy": float(matrix.trace() / matrix.sum()),
                        "class_recall": (matrix.diagonal() / totals).tolist(),
                        "confusion_matrix": matrix.tolist()})
    low = [p["balanced_accuracy"] for p in per_snr if p["requested_snr_db"] in (-20, -25, -30)]
    return {"classes": list(CLASSES), "samples": int(sum(m.sum() for m in matrices.values())),
            "mean_balanced_accuracy": float(np.mean([p["balanced_accuracy"] for p in per_snr])),
            "lowest_three_mean_balanced_accuracy": float(np.mean(low)) if len(low) == 3 else None,
            "per_snr": per_snr}


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.inference_mode()
def evaluate(model, loader, device, snrs=ALL_SNRS):
    model.eval()
    matrices = {int(s): np.zeros((7, 7), dtype=np.int64) for s in snrs}
    synchronize(device)
    start = time.perf_counter()
    for signals, labels, batch_snrs in loader:
        prediction = model(signals.to(device, non_blocking=True)).argmax(1).cpu().numpy()
        labels, batch_snrs = labels.numpy(), batch_snrs.numpy()
        for snr in snrs:
            mask = batch_snrs == snr
            np.add.at(matrices[int(snr)], (labels[mask], prediction[mask]), 1)
    synchronize(device)
    seconds = time.perf_counter() - start
    report = report_from_matrices(matrices)
    report.update(inference_seconds=seconds, seconds_per_pulse=seconds / report["samples"],
                  timing_scope="DataLoader, disk reads, transfer, RMS/noise normalization, both STFTs, CNN, metric accumulation; FP32")
    return report


def hardware_info(device):
    return {"device": str(device), "cpu": platform.processor(), "platform": platform.platform(),
            "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
            "gpu_memory_bytes": torch.cuda.get_device_properties(device).total_memory if device.type == "cuda" else None,
            "python": sys.version, "numpy": str(np.__version__), "torch": str(torch.__version__),
            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version()}


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


# %% Training, checkpoint selection and reproducible freeze bundle
DEFAULT_CONFIG = {
    "seed": 42, "train_pulses_per_file": 80, "train_snrs": list(ALL_SNRS),
    "selection_pulses_per_file": 4, "batch_size": 64, "workers": 2,
    "epochs": 20, "samples_per_epoch": 105600, "patience": 6,
    "learning_rate": 0.0006, "weight_decay": 0.01, "label_smoothing": 0.02,
    "train_amp": True,
    "model_kwargs": {"n_ffts": [256, 1024], "hop_length": 64,
                     "image_size": [256, 256], "dropout": 0.2},
}


def train_and_freeze(data_root, manifest_path, output, config=None, prepared=None):
    cfg = json.loads(json.dumps(DEFAULT_CONFIG if config is None else config))
    output = Path(output)
    # Never overwrite frozen artifacts during a rerun or a new experiment.
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"Output is not empty: {output}. Choose a new RUN_NAME for a new experiment.")
    output.mkdir(parents=True, exist_ok=True)
    if not 1 <= cfg["selection_pulses_per_file"] <= 20:
        raise ValueError("Selection validation uses 1..20 published validation pulses per recording")
    if cfg["epochs"] < 1 or cfg["samples_per_epoch"] < 7:
        raise ValueError("Need at least one epoch and seven samples per epoch")
    seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    setup_start = time.perf_counter()
    manifest_path = find_manifest(data_root, manifest_path)
    if prepared is None:
        rows, infos = prepare_development(data_root, manifest_path)
        preparation_seconds = 0.0
    else:
        rows, infos, preparation_seconds = prepared
    train_ex = build_examples(rows, infos, "train", cfg["train_pulses_per_file"], cfg["train_snrs"])
    select_ex = build_examples(rows, infos, "validation", cfg["selection_pulses_per_file"], ALL_SNRS)
    full_ex = build_examples(rows, infos, "validation", 20, ALL_SNRS)
    train_ids = {(e[0], e[1]) for e in train_ex}
    assert not train_ids.intersection((e[0], e[1]) for e in full_ex), "Source-pulse leakage"
    labels = np.array([e[3] for e in train_ex], dtype=np.int64)
    counts = np.bincount(labels, minlength=7)
    if (counts == 0).any():
        raise ValueError("All seven classes must be present in training")
    sampler = WeightedRandomSampler(torch.as_tensor(1.0 / counts[labels], dtype=torch.double),
                                    cfg["samples_per_epoch"], replacement=True,
                                    generator=torch.Generator().manual_seed(cfg["seed"]))
    train_loader = make_loader(DevelopmentDataset(train_ex, infos), cfg["batch_size"], cfg["workers"], cfg["seed"], sampler)
    select_loader = make_loader(DevelopmentDataset(select_ex, infos), cfg["batch_size"], cfg["workers"], cfg["seed"])
    print("Eligible training examples:", len(train_ex), "| sampled per epoch:", cfg["samples_per_epoch"], flush=True)
    print("Eligible training class counts:", dict(zip(CLASSES, counts.tolist())), flush=True)
    print("Selection validation:", len(select_ex), "| final full validation:", len(full_ex), flush=True)
    model = Track2Model(**cfg["model_kwargs"]).to(device)
    parameter_count = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    use_amp = bool(cfg["train_amp"] and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    best_score, best_low, best_epoch, stale = -1.0, -1.0, 0, 0
    history, fit_seconds, selection_seconds = [], 0.0, 0.0
    setup_seconds = time.perf_counter() - setup_start + preparation_seconds
    synchronize(device)
    start = time.perf_counter()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        epoch_start = time.perf_counter()
        total_loss, seen = 0.0, 0
        for step, (signals, targets, _) in enumerate(train_loader, 1):
            signals, targets = signals.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(signals), targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach()) * len(targets)
            seen += len(targets)
            if step == 1 or step % 100 == 0:
                print(f"Epoch {epoch}/{cfg['epochs']}, batch {step}/{len(train_loader)}, loss {total_loss/seen:.4f}", flush=True)
        synchronize(device)
        fit_seconds += time.perf_counter() - epoch_start
        report = evaluate(model, select_loader, device)
        selection_seconds += report["inference_seconds"]
        score, low = report["mean_balanced_accuracy"], report["lowest_three_mean_balanced_accuracy"]
        record = {"epoch": epoch, "train_loss": total_loss / seen,
                  "learning_rate": optimizer.param_groups[0]["lr"],
                  "selection_mean_balanced_accuracy": score, "selection_lowest_three": low}
        history.append(record)
        print(json.dumps(record), flush=True)
        better = score > best_score + 1e-6 or (abs(score - best_score) <= 1e-6 and low > best_low + 1e-6)
        if better:
            best_score, best_low, best_epoch, stale = score, low, epoch, 0
            torch.save({"state_dict": model.state_dict(), "classes": list(CLASSES),
                        "track": "B", "model_kwargs": cfg["model_kwargs"]}, output / "model.pt")
            report.update(selected_epoch=epoch, pulses_per_recording=cfg["selection_pulses_per_file"],
                          full_published_validation=cfg["selection_pulses_per_file"] == 20)
            write_json(output / "selection_validation.json", report)
        else:
            stale += 1
        write_json(output / "training_history.json", history)
        scheduler.step()
        if stale >= cfg["patience"]:
            print("Early stopping on validation only.", flush=True)
            break
    synchronize(device)
    development_seconds = time.perf_counter() - start
    # Evaluate exactly the saved best checkpoint, never the final in-memory epoch.
    checkpoint = torch.load(output / "model.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    del checkpoint, optimizer, train_loader, select_loader
    full_loader = make_loader(DevelopmentDataset(full_ex, infos), cfg["batch_size"], cfg["workers"], cfg["seed"])
    full_report = evaluate(model, full_loader, device)
    assert full_report["samples"] == 171600
    full_report.update(selected_epoch=best_epoch, pulses_per_recording=20, full_published_validation=True)
    write_json(output / "validation_report.json", full_report)
    cfg.update(track="B", track_number=2, classes=list(CLASSES), sample_rate_hz=SAMPLE_RATE_HZ,
               manifest_sha256=MANIFEST_SHA256, train_source_pulses=len(train_ids),
               eligible_training_examples=len(train_ex), eligible_class_counts=dict(zip(CLASSES, counts.tolist())),
               subset_rule="First N ascending published pulse indices per recording, separately within each split",
               sampler="Inverse-class-count weighted replacement sampling; every selected SNR has equal sampling probability within each class",
               optimizer="AdamW", schedule="CosineAnnealingLR by epoch", gradient_clip_norm=5.0,
               selection_rule="Mean balanced accuracy across all 13 SNRs on selection validation; tolerance 1e-6, lowest-three-SNR tie-break",
               selected_epoch=best_epoch, epochs_completed=len(history), training_draws=len(history)*cfg["samples_per_epoch"],
               inference_precision="float32", inference_augmentation="none", training_augmentation="none beyond supplied SNR variants",
               model_parameters=parameter_count, model_bytes=(output / "model.pt").stat().st_size,
               setup_seconds=setup_seconds, fit_seconds=fit_seconds, selection_validation_seconds=selection_seconds,
               development_seconds=development_seconds, hardware=hardware_info(device),
               preprocessing="Complex RMS per pulse; centered two-sided Hann STFT with zero padding; log1p(power / per-pulse noise-window mean power); frequency shift; adaptive mean pooling",
               pretrained=False, external_training_data=False)
    write_json(output / "config.json", cfg)
    shutil.copyfile(Path(__file__), output / "radarml_track2_runtime.py")
    shutil.copyfile(manifest_path, output / "split_manifest.csv")
    # Public version pins accept the CUDA build already installed on Kaggle.
    # Exact build, CUDA/cuDNN and hardware are retained in config.json.
    torch_public_version = str(torch.__version__).split("+", 1)[0]
    (output / "requirements.txt").write_text(
        f"# Exact CUDA/build details: config.json; use the original Kaggle environment.\n"
        f"numpy=={np.__version__}\ntorch=={torch_public_version}\n", encoding="utf-8")
    (output / "inference_command.txt").write_text(
        "python radarml_track2_runtime.py infer --bundle . --blind-root /path/to/blind_test_participants --output /path/to/predictions.csv\n", encoding="utf-8")
    summary = f"""RadarML Lite — Track 2 / B (time-frequency)

Method: two-sided Hann STFT at {cfg['model_kwargs']['n_ffts']} samples, hop
{cfg['model_kwargs']['hop_length']}; per-pulse RMS/noise-window normalization,
log power and adaptive pooling; a residual 2D CNN with GroupNorm, trained
from scratch with class-balanced replacement sampling, AdamW and cosine LR.
No filename, ID, metadata, pulse index or requested-SNR label is a model input.

Published split: {MANIFEST_SHA256}. Source-pulse training subset:
{len(train_ids):,} ({cfg['train_pulses_per_file']} / 80 per recording), SNRs
{cfg['train_snrs']}. All variants retain the source pulse's split. Seed:
{cfg['seed']}. Draws per epoch: {cfg['samples_per_epoch']:,}; this sampling
budget does not guarantee every eligible variant is visited each epoch.
Training draws actually used: {cfg['training_draws']:,}.

Checkpoint selected on {cfg['selection_pulses_per_file']} validation pulses
per recording at all 13 SNRs; selected epoch: {best_epoch}. After selection,
the saved checkpoint was evaluated on all 171,600 published validation
examples: mean balanced accuracy {full_report['mean_balanced_accuracy']:.6f};
lowest-three-SNR mean {full_report['lowest_three_mean_balanced_accuracy']:.6f}.
See validation_report.json for per-SNR scores and all confusion matrices.
Validation pulses were never fitted or added to final training.

Hardware/dependency versions: config.json and requirements.txt. Model:
{parameter_count:,} parameters, {cfg['model_bytes']:,} bytes. Fit-only time:
{fit_seconds:.1f} s; fit plus selection/checkpoint overhead:
{development_seconds:.1f} s; dataset setup: {setup_seconds:.1f} s.
Full-validation inference: {full_report['seconds_per_pulse']:.6f} s/pulse,
including loading, transfers, normalization, STFTs and CNN. Blind-run timing
is recorded separately in predictions.csv.inference.json after official release.

Limitations: validation and blind test use known waveform configurations;
results do not establish generalization to other receivers or configurations.
Requested SNR differs from achieved SNR, and int16 clipping affects low SNRs.
Pooling trades spectral/time detail for speed. No blind score is known here.

Attribution: Ritchie, White and Hosford (2025), RadarML v1,
https://doi.org/10.5522/04/30752767.v1; organizer Radar_ML_lite code and rules,
GPL-3.0; file-identity lookup adapted from the participant's synapseon.ipynb.
AI assistance: OpenAI Codex helped inspect documentation and implement/debug
the notebook and runtime. No external learned weights or training data used.

Before submitting: identify your team/name and organizer-issued pipeline
identifier in the submission message; keep the hashed bundle unchanged.
Send this frozen bundle at the announced freeze deadline; after blind release
run the inference command, then add predictions.csv and its inference report.
"""
    (output / "method_summary.txt").write_text(summary, encoding="utf-8")
    hashes = {p.name: lookup_sha256(p) for p in sorted(output.iterdir()) if p.is_file()}
    write_json(output / "frozen.sha256.json", hashes)
    print("Full validation mean balanced accuracy:", full_report["mean_balanced_accuracy"], flush=True)
    print("Frozen artifacts:", output, flush=True)
    return full_report


# %% Blind inference: frozen code/config/weights, anonymous shard loader
def verify_bundle(bundle):
    bundle = Path(bundle)
    hashes = json.loads((bundle / "frozen.sha256.json").read_text(encoding="utf-8"))
    required = {"model.pt", "config.json", "radarml_track2_runtime.py", "requirements.txt"}
    if not required.issubset(hashes):
        raise ValueError("Incomplete freeze manifest")
    for name, expected in hashes.items():
        if Path(name).name != name or lookup_sha256(bundle / name) != expected:
            raise ValueError(f"Frozen artifact changed or invalid: {name}")
    # The function actually executing must also be the frozen runtime.
    if lookup_sha256(Path(__file__)) != hashes["radarml_track2_runtime.py"]:
        raise ValueError("Execute the runtime inside the frozen bundle")
    return hashes


def read_blind_index(blind_root, expected_count=171600):
    root = Path(blind_root).resolve()
    if not (root / "index.csv").is_file() and (root / "test" / "index.csv").is_file():
        root = root / "test"
    with (root / "index.csv").open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not {"sample_id", "file", "row"}.issubset(reader.fieldnames or []):
            raise ValueError("Anonymous index must provide sample_id,file,row")
        rows = list(reader)
    template = root / "submission_template.csv"
    if not template.is_file():
        template = root.parent / "submission_template.csv"
    with template.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["sample_id", "predicted_label"]:
            raise ValueError("Unexpected submission template columns")
        ids = [r["sample_id"] for r in reader]
    index_ids = [r["sample_id"] for r in rows]
    if (len(rows) != expected_count or any(not i for i in index_ids) or
            len(set(index_ids)) != len(index_ids) or len(set(ids)) != len(ids) or set(index_ids) != set(ids)):
        raise ValueError("Blind index/template must contain every expected anonymous ID once")
    locations = set()
    for row in rows:
        relative = Path(row["file"])
        path = (root / relative).resolve()
        if relative.is_absolute() or not path.is_relative_to(root):
            raise ValueError("Shard path escapes the blind directory")
        row["row"] = int(row["row"])
        if row["row"] < 0 or (row["file"], row["row"]) in locations:
            raise ValueError("Duplicate or invalid blind shard row")
        locations.add((row["file"], row["row"]))
    return root, rows, ids


class BlindDataset(Dataset):
    def __init__(self, root, rows):
        self.root, self.rows = Path(root), rows

    @staticmethod
    @lru_cache(maxsize=8)
    def _array(path):
        array = np.load(path, mmap_mode="r", allow_pickle=False)
        if array.ndim != 2 or array.shape[1] != 33600 or array.dtype != np.int16:
            raise ValueError(f"Expected int16 blind shard (N,33600): {path}")
        return array

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        info = self.rows[index]
        row = self._array(str(self.root / info["file"]))[info["row"]]
        return torch.from_numpy(interleaved_to_channels(row))


def predict_blind(bundle, blind_root, output, batch_size=64, workers=2):
    bundle, output = Path(bundle), Path(output)
    hashes = verify_bundle(bundle)
    cfg = json.loads((bundle / "config.json").read_text(encoding="utf-8"))
    if cfg["track"] != "B" or cfg["classes"] != list(CLASSES) or cfg["inference_precision"] != "float32":
        raise ValueError("Unexpected frozen pipeline settings")
    seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(bundle / "model.pt", map_location="cpu", weights_only=True)
    if (checkpoint["track"] != "B" or checkpoint["classes"] != list(CLASSES) or
            checkpoint["model_kwargs"] != cfg["model_kwargs"]):
        raise ValueError("Checkpoint disagrees with frozen configuration")
    model = Track2Model(**cfg["model_kwargs"]).to(device)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    del checkpoint
    root, rows, template_ids = read_blind_index(blind_root)
    # Storage-only ordering; IDs are used solely to put outputs into the template.
    rows.sort(key=lambda r: (r["file"], r["row"]))
    loader = make_loader(BlindDataset(root, rows), batch_size, workers, cfg["seed"])
    synchronize(device)
    start = time.perf_counter()
    predictions = []
    with torch.inference_mode():
        for step, signals in enumerate(loader, 1):
            predictions.extend(model(signals.to(device, non_blocking=True)).argmax(1).cpu().tolist())
            if step == 1 or step % 100 == 0:
                print(f"Blind inference: {len(predictions):,}/{len(rows):,}", flush=True)
    synchronize(device)
    elapsed = time.perf_counter() - start
    if len(predictions) != len(rows):
        raise RuntimeError("Incomplete blind inference")
    by_id = {r["sample_id"]: CLASSES[p] for r, p in zip(rows, predictions)}
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + ".partial")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(("sample_id", "predicted_label"))
        writer.writerows((sid, by_id[sid]) for sid in template_ids)
    # Check the actual serialized file before publishing it as predictions.csv.
    with temporary.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["sample_id", "predicted_label"]
        saved = list(reader)
    assert [r["sample_id"] for r in saved] == template_ids
    assert all(r["predicted_label"] in CLASSES for r in saved)
    temporary.replace(output)
    report = {"samples": len(rows), "inference_seconds": elapsed, "seconds_per_pulse": elapsed / len(rows),
              "timing_scope": "Cold DataLoader startup, shard loading, transfers, normalization, both STFTs, CNN and prediction collection; excludes model/index loading and CSV writing",
              "precision": "float32", "hardware": hardware_info(device),
              "frozen_artifact_hashes": hashes, "predictions_sha256": lookup_sha256(output),
              "batch_size": batch_size, "workers": workers}
    write_json(output.with_name(output.name + ".inference.json"), report)
    print("Predictions:", output, "| seconds/pulse:", report["seconds_per_pulse"], flush=True)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    infer = commands.add_parser("infer", help="Inference after official blind-test release only")
    infer.add_argument("--bundle", required=True)
    infer.add_argument("--blind-root", required=True)
    infer.add_argument("--output", required=True)
    infer.add_argument("--batch-size", type=int, default=64)
    infer.add_argument("--workers", type=int, default=2)
    args = parser.parse_args()
    predict_blind(args.bundle, args.blind_root, args.output, args.batch_size, args.workers)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile radarml_track2_neural_infer.py
"""Compact Track 2 neural inference: complex STFT -> neural network -> seven labels.
SPDX-License-Identifier: GPL-3.0-or-later. AI coding assistance: OpenAI Codex.
At inference only this code, model.pt and the blind participant package are needed.
No training examples, waveform-template bank, teacher or previous predictions.
"""
import argparse
import csv
from functools import lru_cache
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import sys
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

CLASSES = ("barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad")
MODEL_KWARGS = dict(n_fft=256, projection_rank=64, hidden=128, dropout=.1)


class ComplexSTFT(nn.Module):
    """Explicit frequency x time grid, preserving phase and all 16,800 input samples."""
    def __init__(self, n_fft=256):
        super().__init__()
        self.n_fft = n_fft
        self.frames = math.ceil(16800 / n_fft)
        self.register_buffer("window", torch.ones(n_fft), persistent=False)

    def forward(self, iq):
        with torch.autocast(device_type=iq.device.type, enabled=False):
            iq = iq.float()
            power = iq.square().sum(1)
            rms2 = power.mean(1, keepdim=True).clamp_min(1e-10)
            z = torch.complex(iq[:, 0], iq[:, 1]) / rms2.sqrt()
            z = F.pad(z, (0, self.frames * self.n_fft - z.shape[-1]))
            stft = torch.stft(z, n_fft=self.n_fft, hop_length=self.n_fft,
                              win_length=self.n_fft, window=self.window, normalized=True,
                              center=False, onesided=False, return_complex=True)
            stft = torch.fft.fftshift(stft, dim=1)
            # Non-overlapping rectangular STFT blocks form a unitary block transform:
            # the two-channel complex view retains information and an explicit time axis.
            phase = torch.stack((stft.real, stft.imag), 1)
            noise = power[:, :2400].mean(1, keepdim=True).clamp_min(1e-10)
            relative_noise = (noise / rms2).clamp_min(1e-6)
            image = torch.log1p(stft.abs().square() / relative_noise[:, :, None]).unsqueeze(1)
            active = power[:, 2520:14280].mean(1, keepdim=True)
            tail = power[:, 14640:].mean(1, keepdim=True)
            clipped = ((iq <= -1.) | (iq >= 32767. / 32768.)).float().mean((1, 2))[:, None]
            stats = torch.cat((relative_noise.log(), torch.log1p(active / noise),
                               torch.log1p(tail / noise), clipped), 1)
            return phase, image, stats


class SeparableBlock(nn.Module):
    def __init__(self, incoming, outgoing, stride=(2, 2)):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(incoming, incoming, 3, stride, 1, groups=incoming, bias=False),
            nn.GroupNorm(4, incoming), nn.SiLU(),
            nn.Conv2d(incoming, outgoing, 1, bias=False),
            nn.GroupNorm(4, outgoing), nn.SiLU())

    def forward(self, x):
        return self.layers(x)


class CompactSTFTClassifier(nn.Module):
    """Learned low-rank phase features + small power-image CNN; no configuration search."""
    def __init__(self, n_fft=256, projection_rank=64, hidden=128, dropout=.1):
        super().__init__()
        self.representation = ComplexSTFT(n_fft)
        features = 2 * n_fft * self.representation.frames
        if projection_rank < 8 or projection_rank % 2:
            raise ValueError("projection_rank must be an even number >= 8")
        global_rank = projection_rank // 2
        self.phase_global = nn.Sequential(nn.Flatten(), nn.Linear(features, global_rank),
                                           nn.LayerNorm(global_rank), nn.SiLU())
        local_features = 8 * math.ceil(n_fft / 4) * math.ceil(self.representation.frames / 2)
        self.phase_local = nn.Sequential(
            nn.Conv2d(2, 8, 3, (2, 1), 1, bias=False), nn.GroupNorm(4, 8), nn.SiLU(),
            SeparableBlock(8, 8, (2, 2)), nn.Flatten(), nn.Linear(local_features, projection_rank),
            nn.LayerNorm(projection_rank), nn.SiLU())
        self.phase_merge = nn.Sequential(nn.Linear(global_rank + projection_rank, hidden),
                                          nn.LayerNorm(hidden), nn.SiLU())
        self.power_branch = nn.Sequential(
            nn.Conv2d(1, 12, (5, 3), (2, 1), (2, 1), bias=False),
            nn.GroupNorm(4, 12), nn.SiLU(),
            SeparableBlock(12, 24), SeparableBlock(24, 40), SeparableBlock(40, 64),
            nn.AdaptiveAvgPool2d((8, 4)), nn.Flatten(), nn.Linear(64 * 8 * 4, hidden), nn.SiLU())
        self.head = nn.Sequential(nn.Linear(2 * hidden + 4, hidden), nn.SiLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, len(CLASSES)))

    def forward(self, iq, return_embedding=False):
        phase, power, stats = self.representation(iq)
        phase_features = self.phase_merge(torch.cat((self.phase_global(phase), self.phase_local(phase)), 1))
        embedding = torch.cat((phase_features, self.power_branch(power), stats), 1)
        logits = self.head(embedding)
        return (logits, embedding) if return_embedding else logits


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def load_model(path, device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if (checkpoint.get("model_family") != "compact_complex_stft_neural" or
            checkpoint.get("classes") != list(CLASSES) or checkpoint.get("track") != "B"):
        raise ValueError("This script requires the compact STFT neural model, not a template bank")
    model = CompactSTFTClassifier(**checkpoint["model_kwargs"])
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    return model.to(device).eval(), checkpoint


def find_participants(explicit=None, search_root=Path("/kaggle/input")):
    root = Path(explicit if explicit is not None else search_root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Participant folder does not exist: {root}")
    if (root / "index.csv").is_file() and (root.parent / "submission_template.csv").is_file():
        return root.parent
    candidates = {p.parent for p in root.rglob("submission_template.csv")
                  if (p.parent / "test/index.csv").is_file() or (p.parent / "index.csv").is_file()}
    if len(candidates) != 1:
        raise ValueError(f"Expected one participant package under {root}; found {len(candidates)}. Set BLIND_ROOT explicitly.")
    return candidates.pop()


def read_index(participants, expected_count=171600):
    root = Path(participants).resolve()
    if (root / "test/index.csv").is_file():
        root = root / "test"
    with (root / "index.csv").open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["sample_id", "file", "row"]:
            raise ValueError("Expected anonymous index columns: sample_id,file,row")
        rows = list(reader)
    template = root / "submission_template.csv"
    if not template.exists():
        template = root.parent / "submission_template.csv"
    with template.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames not in (["sample_id", "predicted_label"], ["sample_id", "label"]):
            raise ValueError("Unexpected submission template columns")
        ids = [r["sample_id"] for r in reader]
    index_ids = [r["sample_id"] for r in rows]
    if (len(rows) != expected_count or len(set(index_ids)) != len(rows) or
            len(set(ids)) != len(ids) or set(ids) != set(index_ids) or any(not sid for sid in ids)):
        raise ValueError("Index/template IDs must match and contain every expected sample once")
    locations = set()
    for row in rows:
        path = (root / row["file"]).resolve()
        if not path.is_relative_to(root) or Path(row["file"]).is_absolute():
            raise ValueError("Shard path escapes the participant folder")
        row["row"] = int(row["row"])
        if row["row"] < 0 or (row["file"], row["row"]) in locations:
            raise ValueError("Invalid/duplicate shard row")
        locations.add((row["file"], row["row"]))
    rows.sort(key=lambda r: (r["file"], r["row"]))
    return root, rows, ids


class BlindSignals(Dataset):
    def __init__(self, root, rows):
        self.root, self.rows = Path(root), rows

    @staticmethod
    @lru_cache(maxsize=8)
    def array(path):
        value = np.load(path, mmap_mode="r", allow_pickle=False)
        if value.ndim != 2 or value.shape[1] != 33600 or value.dtype != np.int16:
            raise ValueError(f"Expected int16 shard (N,33600): {path}")
        return value

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        info = self.rows[index]
        raw = self.array(str(self.root / info["file"]))[info["row"]]
        iq = np.stack((raw[::2], raw[1::2])).astype(np.float32) / 32768.
        return torch.from_numpy(iq)


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def hardware(device):
    return dict(device=str(device), cpu=platform.processor(), logical_cpus=os.cpu_count(),
                threads=torch.get_num_threads(), platform=platform.platform(), python=sys.version,
                torch=str(torch.__version__), numpy=str(np.__version__),
                gpu=torch.cuda.get_device_name(device) if device.type == "cuda" else None)


def latency_stats(times):
    a = np.asarray(times)
    return dict(samples=len(a), mean_ms=float(a.mean()), median_ms=float(np.median(a)),
                p95_ms=float(np.percentile(a, 95)), p99_ms=float(np.percentile(a, 99)),
                max_ms=float(a.max()), fraction_at_or_below_10ms=float(np.mean(a <= 10.)))


@torch.inference_mode()
def predict(model_path, participants, output, device_name="cpu", batch_size=128,
            workers=0, threads=4, benchmark_samples=1000, warmup=20, label_column="label",
            expected_count=171600):
    start = time.perf_counter()
    model_path, output = Path(model_path).resolve(), Path(output).resolve()
    participants = find_participants(participants)
    if output.is_relative_to(participants) or output == model_path:
        raise ValueError("Choose an output outside the model and blind input files")
    if label_column not in {"label", "predicted_label"}:
        raise ValueError("Invalid label column")
    if min(batch_size, threads, benchmark_samples) < 1 or min(workers, warmup) < 0:
        raise ValueError("Invalid inference settings")
    torch.set_num_threads(threads)
    device = torch.device(device_name)
    if device.type not in {"cpu", "cuda"} or (device.type == "cuda" and not torch.cuda.is_available()):
        raise ValueError("Choose an available cpu or cuda device")
    tick = time.perf_counter()
    model, checkpoint = load_model(model_path, device)
    synchronize(device)
    setup_seconds = time.perf_counter() - tick
    root, rows, template_ids = read_index(participants, expected_count)
    dataset = BlindSignals(root, rows)
    indices = np.linspace(0, len(rows) - 1, min(benchmark_samples, len(rows)), dtype=np.int64)
    for i in range(warmup):
        signal = dataset[int(indices[i % len(indices)])][None].to(device)
        model(signal).argmax(1).item()
    synchronize(device)
    latencies = []
    for index in indices:
        tick = time.perf_counter()
        signal = dataset[int(index)][None].to(device)
        model(signal).argmax(1).item()
        synchronize(device)
        latencies.append((time.perf_counter() - tick) * 1000.)
    single = latency_stats(latencies)
    print("Single-signal latency, including I/Q conversion and STFT:", single, flush=True)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=workers, shuffle=False,
                        pin_memory=device.type == "cuda")
    predictions = []
    synchronize(device)
    tick = time.perf_counter()
    for step, signals in enumerate(loader, 1):
        predictions.extend(model(signals.to(device, non_blocking=True)).argmax(1).cpu().tolist())
        if step == 1 or step % 100 == 0:
            print(f"Inference: {len(predictions):,}/{len(rows):,}", flush=True)
    synchronize(device)
    seconds = time.perf_counter() - tick
    if len(predictions) != len(rows):
        raise RuntimeError("Incomplete predictions")
    by_id = {r["sample_id"]: CLASSES[p] for r, p in zip(rows, predictions)}
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + ".partial")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["sample_id", label_column])
        writer.writerows((sid, by_id[sid]) for sid in template_ids)
    with temporary.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        saved = list(reader)
        if (reader.fieldnames != ["sample_id", label_column] or
                [r["sample_id"] for r in saved] != template_ids or
                any(r[label_column] not in CLASSES for r in saved)):
            raise ValueError("Invalid serialized submission")
    temporary.replace(output)
    report = dict(samples=len(rows), model_bytes=model_path.stat().st_size,
                  model_parameters=sum(p.numel() for p in model.parameters()), model_setup_seconds=setup_seconds,
                  inference_seconds=seconds, milliseconds_per_signal_throughput=1000. * seconds / len(rows),
                  single_signal_latency=single, hardware=hardware(device), precision="float32",
                  timing_scope="Single-signal and batch timing include loading/conversion, transfer if applicable, "
                               "full complex STFT, both neural branches and class prediction; exclude one-time model loading. "
                               "OS cache state is uncontrolled. Single-signal warm-up is excluded.",
                  model_sha256=sha256(model_path), inference_code_sha256=sha256(Path(__file__)),
                  predictions_sha256=sha256(output), columns=["sample_id", label_column],
                  batch_size=batch_size, workers=workers, warmup_signals=warmup,
                  total_seconds=time.perf_counter() - start)
    write_json(output.with_name(output.name + ".inference.json"), report)
    print("Submission:", output, flush=True)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", type=Path, required=True)
    parser.add_argument("--participants", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--workers", type=int, default=0)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--benchmark-samples", type=int, default=1000)
    parser.add_argument("--warmup", type=int, default=20)
    parser.add_argument("--label-column", choices=["label", "predicted_label"], default="label")
    args = parser.parse_args()
    predict(args.model, args.participants, args.output, args.device, args.batch_size,
            args.workers, args.threads, args.benchmark_samples, args.warmup, args.label_column)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile radarml_track2_neural_train.py
"""Development-only trainer for the compact complex-STFT neural classifier.
SPDX-License-Identifier: GPL-3.0-or-later. AI coding assistance: OpenAI Codex.
This module and its training cache are NOT required or exported for inference.
"""
from collections import defaultdict
import copy
import csv
from functools import lru_cache
import json
import math
from pathlib import Path
import shutil
import time
import zipfile

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import radarml_track2_runtime as helpers
import radarml_track2_neural_infer as infer

DEFAULT_CONFIG = dict(seed=42, epochs=80, samples_per_epoch=105600, batch_size=256, workers=2,
                      learning_rate=.001, weight_decay=.001, label_smoothing=.005,
                      clean_loss_weight=.25, feature_consistency_weight=2.,
                      ema_decay=.999, warmup_epochs=2, patience=18, max_run_hours=10.5,
                      train_pulses_per_file=80, selection_pulses_per_file=4,
                      train_snrs=list(helpers.ALL_SNRS), hard_snr_weight=2.,
                      model_kwargs=dict(infer.MODEL_KWARGS))


def find_data_root(explicit=None, kind="auto", search_root=Path("/kaggle/input")):
    root = Path(explicit if explicit is not None else search_root).expanduser().resolve()
    if kind not in {"auto", "compact", "expanded"} or not root.is_dir():
        raise ValueError(f"Set an existing DATA_ROOT and valid DATA_FORMAT; searched {root}")
    counts = defaultdict(int)
    for path in root.rglob("*.npy"):
        name = helpers.canonical_filename(path.name)
        fmt = ("compact" if name.endswith("_recorded.npy") else
               "expanded" if name.endswith("_single_channel.npy") else None)
        if fmt is None or (kind != "auto" and kind != fmt):
            continue
        for parent in path.parents:
            if not parent.is_relative_to(root):
                break
            counts[(parent, fmt)] += 1
    complete = [(p, f) for (p, f), count in counts.items() if count == 660]
    complete = [(p, f) for p, f in complete if not any(q != p and q.is_relative_to(p) and g == f for q, g in complete)]
    if len(complete) != 1:
        choices = "\n".join(f"{p} ({f})" for p, f in complete)
        folders = "\n".join(str(p) for p in sorted(root.iterdir()) if p.is_dir())
        raise ValueError(f"Expected one complete 660-recording development dataset under {root}; found {len(complete)}. "
                         f"Set DATA_ROOT/DATA_FORMAT explicitly if needed.\nCandidates:\n{choices}\nFolders:\n{folders}")
    return complete[0][0]


def prepare(root, manifest, kind):
    root = Path(root)
    if kind == "auto":
        compact = any(helpers.canonical_filename(p.name).endswith("_recorded.npy") for p in root.rglob("*.npy"))
        expanded = any(helpers.canonical_filename(p.name).endswith("_single_channel.npy") for p in root.rglob("*.npy"))
        if compact == expanded:
            raise ValueError("Set DATA_FORMAT to choose one development format")
        kind = "compact" if compact else "expanded"
    if kind == "expanded":
        rows, infos = helpers.prepare_development(root, manifest)
        for info in infos.values():
            info["kind"] = kind
        return rows, infos
    if kind != "compact":
        raise ValueError("DATA_FORMAT must be auto, compact or expanded")
    rows = helpers.load_manifest(manifest)
    lookup = helpers.build_file_lookup(root, "compact")
    infos, used = {}, set()
    for index, name in enumerate(sorted({r["file"] for r in rows}), 1):
        path = helpers.resolve_file(name, lookup, "compact")
        sidecar, meta = helpers.resolve_metadata(name, path, lookup, "compact")
        expected = name.replace("_recorded.npy", "_single_channel.npy")
        if helpers.canonical_filename(helpers.expanded_name_from_source(meta["source_file"])) != helpers.canonical_filename(expected):
            raise ValueError(f"Wrong source identity: {sidecar}")
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        if (arr.shape != (1, 100, 33600) or arr.dtype != np.int16 or
                meta.get("format") != "radarml-recorded-v1" or meta.get("source_pulse_indices") != list(range(100)) or
                meta.get("sample_rate_hz") != 120000000 or meta.get("channel") != "ADC0" or
                path in used or helpers.lookup_sha256(path) != meta.get("output_sha256")):
            raise ValueError(f"Invalid or corrupted compact recording: {path}")
        used.add(path)
        infos[name] = dict(path=str(path), label=infer.CLASSES.index(helpers.family(name)), kind=kind,
                           source_file=meta["source_file"], snr_to_axis={})
        if index % 100 == 0 or index == 660:
            print(f"Verified compact data: {index}/660", flush=True)
    return rows, infos


def atomic_save(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    torch.save(value, temporary)
    temporary.replace(path)


def build_training_cache(rows, infos, directory, cfg, manifest_hash):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    groups = defaultdict(list)
    validation_ids = set()
    for row in rows:
        if row["split"] == "train":
            groups[row["file"]].append(row["pulse_index"])
        else:
            validation_ids.add((row["file"], row["pulse_index"]))
    selected = [[name, sorted(indices)[:cfg["train_pulses_per_file"]], infos[name]["label"]]
                for name, indices in sorted(groups.items())]
    if any((name, idx) in validation_ids for name, indices, _ in selected for idx in indices):
        raise ValueError("Validation source pulse entered training cache")
    if set(groups) != set(infos):
        raise ValueError("Training file identities do not match the supplied data")
    count = sum(len(indices) for _, indices, _ in selected)
    identity = dict(version=1, manifest_sha256=manifest_hash, source_format=next(iter(infos.values()))["kind"],
                    base_snr=30, records=selected, count=count)
    identity_path = directory / "identity.json"
    if identity_path.exists() and json.loads(identity_path.read_text()) != identity:
        raise ValueError("Training cache identity changed; use a new RUN_NAME")
    helpers.write_json(identity_path, identity)
    raw_path, labels_path = directory / "train_iq.npy", directory / "labels.npy"
    if raw_path.exists() and labels_path.exists():
        cache = np.load(raw_path, mmap_mode="r", allow_pickle=False)
        labels = np.load(labels_path, allow_pickle=False)
        if cache.shape != (count, 33600) or cache.dtype != np.int16 or labels.shape != (count,):
            raise ValueError("Invalid completed training cache")
        print(f"Reusing training-only cache: {count:,} source pulses", flush=True)
        return raw_path, labels_path
    temporary = directory / "train_iq.partial.npy"
    progress_path = directory / "progress.json"
    completed = json.loads(progress_path.read_text())["completed"] if progress_path.exists() and temporary.exists() else 0
    cache = (np.load(temporary, mmap_mode="r+", allow_pickle=False) if completed else
             np.lib.format.open_memmap(temporary, mode="w+", dtype=np.int16, shape=(count, 33600)))
    labels, offset = np.empty(count, np.int64), 0
    for index, (name, indices, label) in enumerate(selected):
        stop = offset + len(indices)
        labels[offset:stop] = label
        if index >= completed:
            info = infos[name]
            arr = np.load(info["path"], mmap_mode="r", allow_pickle=False)
            cache[offset:stop] = (arr[0, indices] if info["kind"] == "compact" else
                                  arr[0, info["snr_to_axis"][30], indices])
            cache.flush()
            helpers.write_json(progress_path, dict(completed=index + 1))
        offset = stop
        if index % 50 == 0 or index + 1 == len(selected):
            print(f"Training-only cache: {index + 1}/{len(selected)} recordings", flush=True)
    cache.flush()
    del cache
    temporary.replace(raw_path)
    np.save(labels_path, labels, allow_pickle=False)
    print(f"Cache size: {raw_path.stat().st_size / 1e9:.2f} GB (development only; never exported)", flush=True)
    return raw_path, labels_path


class CachedTraining(Dataset):
    def __init__(self, path, labels):
        self.path, self.labels = str(path), np.load(labels, allow_pickle=False)
        self.array = None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        if self.array is None:
            self.array = np.load(self.path, mmap_mode="r", allow_pickle=False)
        raw = self.array[index]
        iq = np.stack((raw[::2], raw[1::2])).astype(np.float32) / 32768.
        return torch.from_numpy(iq), int(self.labels[index])


class NoiseAugment(nn.Module):
    """Noise/clipping derived only from each training pulse, across every requested SNR."""
    def __init__(self, snrs, hard_weight=2.):
        super().__init__()
        self.register_buffer("snrs", torch.tensor(snrs, dtype=torch.float32))
        self.register_buffer("probabilities", torch.tensor([hard_weight if s <= -20 else 1. for s in snrs]))

    def forward(self, base, easy=False):
        with torch.autocast(device_type=base.device.type, enabled=False):
            base = base.float()
            power = base.square().sum(1)
            noise = power[:, :2400].mean(1)
            signal = (power[:, 2520:14280].mean(1) - noise).clamp_min(1e-12)
            probabilities = self.probabilities.clone()
            if easy:
                probabilities[self.snrs < -15] = 0
            choice = torch.multinomial(probabilities, len(base), replacement=True)
            snr = self.snrs[choice]
            added = (signal / torch.pow(10., snr / 10.) - noise).clamp_min(0.)
            noisy = base + torch.randn_like(base) * (added / 2.).sqrt()[:, None, None]
            # Match the ADC cap and int16 truncation convention, after independent
            # augmentation noise. Requested SNR stays a target, not a model input.
            noisy = torch.trunc(noisy.clamp(-1., 32767. / 32768.) * 32768.) / 32768.
            return noisy, snr


@lru_cache(maxsize=8)
def mapped(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)


def supplied_noise(raw, snr, source, pulse):
    # Validation-only reconstruction, bitwise equivalent to the supplied compact
    # expander (seed 42), not the independently generated training augmentation.
    import hashlib
    z = raw[::2].astype(np.float64) + 1j * raw[1::2].astype(np.float64)
    noise = np.mean(abs(z[:2400]) ** 2)
    signal = np.mean(abs(z[2520:14280]) ** 2) - noise
    if signal <= 0:
        raise ValueError("Nonpositive source signal power")
    sigma = np.sqrt(max(0., signal / 10. ** (snr / 10.) - noise) / 2.)
    seed = int(hashlib.md5(f"42_{source}_{snr}_{pulse}".encode()).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed)
    z = z + rng.normal(0, sigma, len(z)) + 1j * rng.normal(0, sigma, len(z))
    out = np.empty(raw.shape, np.float64)
    out[::2], out[1::2] = z.real, z.imag
    return np.clip(out, -32768, 32767).astype(np.int16)


class Validation(Dataset):
    def __init__(self, examples, infos, selection_ids=()):
        self.examples, self.infos, self.selection_ids = examples, infos, set(selection_ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        name, pulse, snr, label = self.examples[index]
        info = self.infos[name]
        arr = mapped(info["path"])
        raw = (arr[0, info["snr_to_axis"][snr], pulse] if info["kind"] == "expanded" else
               supplied_noise(arr[0, pulse], snr, info["source_file"], pulse))
        return torch.from_numpy(helpers.interleaved_to_channels(raw)), label, snr, (name, pulse) in self.selection_ids


def loader(dataset, batch_size, workers, seed, sampler=None, cuda=False):
    options = dict(batch_size=batch_size, num_workers=workers, shuffle=False, sampler=sampler,
                   pin_memory=cuda, generator=torch.Generator().manual_seed(seed), worker_init_fn=helpers.seed_worker)
    if workers:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(dataset, **options)


@torch.inference_mode()
def evaluate(models, data_loader, device, audit=False):
    matrices = {name: {s: np.zeros((7, 7), np.int64) for s in helpers.ALL_SNRS} for name in models}
    holdouts = {name: {s: np.zeros((7, 7), np.int64) for s in helpers.ALL_SNRS} for name in models}
    for model in models.values():
        model.eval()
    infer.synchronize(device)
    start, seen = time.perf_counter(), 0
    for step, (signals, labels, snrs, selected) in enumerate(data_loader, 1):
        signals = signals.to(device, non_blocking=True)
        truth, snrs = labels.numpy(), snrs.numpy()
        for name, model in models.items():
            predictions = model(signals).argmax(1).cpu().numpy()
            for snr in helpers.ALL_SNRS:
                mask = snrs == snr
                np.add.at(matrices[name][snr], (truth[mask], predictions[mask]), 1)
                if audit:
                    other = mask & ~selected.numpy()
                    np.add.at(holdouts[name][snr], (truth[other], predictions[other]), 1)
        seen += len(signals)
        if step == 1 or step % 100 == 0:
            print(f"Validation: {seen:,}/{len(data_loader.dataset):,}", flush=True)
    infer.synchronize(device)
    seconds = time.perf_counter() - start
    reports = {name: helpers.report_from_matrices(value) for name, value in matrices.items()}
    for report in reports.values():
        report.update(inference_seconds=seconds, seconds_per_pulse=seconds / seen,
                      timing_scope="Data loading, transfer, complete STFT, neural inference and metrics; FP32")
    return reports, ({name: helpers.report_from_matrices(value) for name, value in holdouts.items()} if audit else {})


def cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def checkpoint_for_export(state, cfg, epoch, variant):
    return dict(state_dict=state, model_kwargs=cfg["model_kwargs"], classes=list(infer.CLASSES), track="B",
                model_family="compact_complex_stft_neural", selected_epoch=epoch, selected_variant=variant,
                seed=cfg["seed"], sample_rate_hz=120000000)


def train(root, manifest, run_dir, config=None, kind="auto", prepared=None):
    cfg = json.loads(json.dumps(DEFAULT_CONFIG if config is None else config))
    if (not 1 <= cfg["train_pulses_per_file"] <= 80 or not 1 <= cfg["selection_pulses_per_file"] < 20 or
            cfg["epochs"] < 1 or cfg["samples_per_epoch"] < 7 or cfg["batch_size"] < 1 or cfg["workers"] < 0 or
            set(cfg["train_snrs"]) != set(helpers.ALL_SNRS) or cfg["max_run_hours"] <= 0):
        raise ValueError("Invalid training settings; all 13 training SNRs must remain available")
    run = Path(run_dir)
    run.mkdir(parents=True, exist_ok=True)
    config_path = run / "run_config.json"
    if config_path.exists() and json.loads(config_path.read_text()) != cfg:
        raise ValueError("Configuration changed; choose a new RUN_NAME")
    helpers.write_json(config_path, cfg)
    export = run / "export"
    if (export / "complete.json").exists():
        old = json.loads((export / "complete.json").read_text())
        if infer.sha256(export / "model.pt") != old["model_sha256"]:
            raise ValueError("Saved model changed after export")
        print("Completed model already exported:", export, flush=True)
        return json.loads((export / "validation_report.json").read_text())
    start = time.perf_counter()
    helpers.seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    manifest = helpers.find_manifest(root, manifest)
    rows, infos = prepared if prepared is not None else prepare(root, manifest, kind)
    cache_raw, cache_labels = build_training_cache(rows, infos, run / "training_cache", cfg, helpers.lookup_sha256(manifest))
    training = CachedTraining(cache_raw, cache_labels)
    labels = training.labels
    counts = np.bincount(labels, minlength=7)
    if (counts == 0).any():
        raise ValueError("Every class must be present in training")
    grouped = defaultdict(list)
    for row in rows:
        if row["split"] == "validation":
            grouped[row["file"]].append(row["pulse_index"])
    selection_ids = {(name, idx) for name, values in grouped.items()
                     for idx in sorted(values)[:cfg["selection_pulses_per_file"]]}
    selection_examples = helpers.build_examples(rows, infos, "validation", cfg["selection_pulses_per_file"], helpers.ALL_SNRS)
    select_loader = loader(Validation(selection_examples, infos), cfg["batch_size"], cfg["workers"], cfg["seed"], cuda=device.type == "cuda")
    model = infer.CompactSTFTClassifier(**cfg["model_kwargs"]).to(device)
    ema = copy.deepcopy(model).eval()
    for parameter in ema.parameters():
        parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    augment = NoiseAugment(cfg["train_snrs"], cfg["hard_snr_weight"]).to(device)
    history, epoch_start, updates, best_score, best_low, stale, fit_seconds = [], 1, 0, -1., -1., 0, 0.
    resume_path = run / "resume.pt"
    if resume_path.exists():
        saved = torch.load(resume_path, map_location="cpu", weights_only=True)
        if saved["config"] != cfg:
            raise ValueError("Resume configuration differs")
        model.load_state_dict(saved["model"])
        ema.load_state_dict(saved["ema"])
        optimizer.load_state_dict(saved["optimizer"])
        scaler.load_state_dict(saved["scaler"])
        epoch_start, updates = saved["next_epoch"], saved["updates"]
        history, best_score, best_low, stale, fit_seconds = saved["history"], saved["best_score"], saved["best_low"], saved["stale"], saved["fit_seconds"]
        torch.set_rng_state(saved["rng_state"])
        if device.type == "cuda" and saved["cuda_rng"]:
            torch.cuda.set_rng_state_all(saved["cuda_rng"])
        print("Resuming at epoch", epoch_start, flush=True)
    print("Trainable parameters:", sum(p.numel() for p in model.parameters()), flush=True)
    print("Training source pulses:", len(training), "| SNRs:", cfg["train_snrs"], flush=True)
    setup_seconds = time.perf_counter() - start
    stop_reason = "epochs_completed"
    for epoch in range(epoch_start, cfg["epochs"] + 1):
        if time.perf_counter() - start >= cfg["max_run_hours"] * 3600:
            stop_reason = "wall_time_budget"
            break
        sampler = WeightedRandomSampler(torch.as_tensor(1. / counts[labels], dtype=torch.double),
                                        cfg["samples_per_epoch"], replacement=True,
                                        generator=torch.Generator().manual_seed(cfg["seed"] + epoch))
        train_loader = loader(training, cfg["batch_size"], cfg["workers"], cfg["seed"] + epoch,
                              sampler, device.type == "cuda")
        model.train()
        tick, seen, loss_total, batches = time.perf_counter(), 0, 0., 0
        snr_counts = {s: 0 for s in cfg["train_snrs"]}
        for step, (base, target) in enumerate(train_loader, 1):
            base, target = base.to(device, non_blocking=True), target.to(device, non_blocking=True)
            noisy, requested_snr = augment(base, easy=epoch <= cfg["warmup_epochs"])
            progress = (epoch - 1 + (step - 1) / len(train_loader)) / cfg["epochs"]
            ramp = min(1., (epoch - 1 + step / len(train_loader)) / max(1, cfg["warmup_epochs"]))
            lr = cfg["learning_rate"] * ramp * (.05 + .95 * (1. + math.cos(math.pi * progress)) / 2.)
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp):
                logits, features = model(noisy, return_embedding=True)
                clean_logits, clean_features = model(base, return_embedding=True)
                loss = F.cross_entropy(logits, target, label_smoothing=cfg["label_smoothing"])
                loss = loss + cfg["clean_loss_weight"] * F.cross_entropy(clean_logits, target)
                consistency = F.mse_loss(F.normalize(features[:, :-4].float(), dim=1),
                                         F.normalize(clean_features[:, :-4].detach().float(), dim=1))
                loss = loss + cfg["feature_consistency_weight"] * consistency
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite training loss; checkpoint from the preceding epoch remains intact")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.)
            scaler.step(optimizer)
            scaler.update()
            updates += 1
            decay = min(cfg["ema_decay"], 1. - 1. / (updates + 1.))
            with torch.no_grad():
                averaged_state = ema.state_dict()
                for key, value in model.state_dict().items():
                    average = averaged_state[key]
                    if value.is_floating_point():
                        average.mul_(decay).add_(value, alpha=1. - decay)
                    else:
                        average.copy_(value)
            seen += len(base)
            loss_total += float(loss.detach()) * len(base)
            batches += 1
            values, frequencies = torch.unique(requested_snr, return_counts=True)
            for value, frequency in zip(values.cpu().tolist(), frequencies.cpu().tolist()):
                snr_counts[int(value)] += frequency
            if step == 1 or step % 100 == 0:
                print(f"Epoch {epoch}/{cfg['epochs']} batch {step}/{len(train_loader)} loss {loss_total / seen:.5f}", flush=True)
            if time.perf_counter() - start >= cfg["max_run_hours"] * 3600:
                stop_reason = "wall_time_budget"
                break
        infer.synchronize(device)
        fit_seconds += time.perf_counter() - tick
        # Release loader workers before running validation. Cache remains memory-mapped.
        del train_loader
        reports, _ = evaluate({"raw": model, "ema": ema}, select_loader, device)
        better_this_epoch = False
        for variant in ("raw", "ema"):
            report = reports[variant]
            score, low = report["mean_balanced_accuracy"], report["lowest_three_mean_balanced_accuracy"]
            better = score > best_score + 1e-6 or (abs(score - best_score) <= 1e-6 and low > best_low + 1e-6)
            if better:
                best_score, best_low = score, low
                best = ema if variant == "ema" else model
                atomic_save(run / "best.pt", checkpoint_for_export(cpu_state(best), cfg, epoch, variant))
                helpers.write_json(run / "best_selection_validation.json", report)
                better_this_epoch = True
        stale = 0 if better_this_epoch else stale + 1
        record = dict(epoch=epoch, train_loss=loss_total / seen, draws=seen, batches=batches,
                      requested_snr_counts={str(k): v for k, v in snr_counts.items()}, learning_rate=lr,
                      raw_selection_score=reports["raw"]["mean_balanced_accuracy"],
                      ema_selection_score=reports["ema"]["mean_balanced_accuracy"], best_score=best_score)
        history.append(record)
        helpers.write_json(run / "history.json", history)
        atomic_save(resume_path, dict(model=cpu_state(model), ema=cpu_state(ema), optimizer=optimizer.state_dict(),
                                      scaler=scaler.state_dict(), config=cfg, next_epoch=epoch + 1, updates=updates,
                                      history=history, best_score=best_score, best_low=best_low, stale=stale,
                                      fit_seconds=fit_seconds, rng_state=torch.get_rng_state(),
                                      cuda_rng=torch.cuda.get_rng_state_all() if device.type == "cuda" else []))
        print(json.dumps(record), flush=True)
        if stop_reason == "wall_time_budget":
            break
        if epoch > cfg["warmup_epochs"] and stale >= cfg["patience"]:
            stop_reason = "early_stopping"
            break
    if not (run / "best.pt").exists():
        raise RuntimeError("No epoch completed; rerun with more wall-time budget or prepare/cache data first")
    selected, selected_checkpoint = infer.load_model(run / "best.pt", device)
    full_examples = helpers.build_examples(rows, infos, "validation", 20, helpers.ALL_SNRS)
    full_loader = loader(Validation(full_examples, infos, selection_ids), cfg["batch_size"], cfg["workers"], cfg["seed"], cuda=device.type == "cuda")
    full, audit = evaluate({"selected": selected}, full_loader, device, audit=True)
    report, audit_report = full["selected"], audit["selected"]
    report.update(full_published_validation=True, selected_epoch=selected_checkpoint["selected_epoch"],
                  selected_variant=selected_checkpoint["selected_variant"])
    audit_report.update(excludes_selection_source_pulses=True)
    export.mkdir(parents=True, exist_ok=True)
    shutil.copy2(run / "best.pt", export / "model.pt")
    shutil.copy2(Path(infer.__file__), export / "radarml_track2_neural_infer.py")
    helpers.write_json(export / "validation_report.json", report)
    helpers.write_json(export / "audit_validation.json", audit_report)
    helpers.write_json(export / "training_config.json", cfg)
    shutil.copy2(run / "history.json", export / "history.json")
    shutil.copy2(run / "best_selection_validation.json", export / "selection_validation.json")
    (export / "requirements.txt").write_text(f"numpy=={np.__version__}\ntorch=={str(torch.__version__).split('+')[0]}\n")
    summary = dict(model_family="compact_complex_stft_neural", classes=list(infer.CLASSES), track="B",
                   parameters=sum(p.numel() for p in selected.parameters()), model_bytes=(export / "model.pt").stat().st_size,
                   training_source_pulses=len(training), eligible_training_snrs=cfg["train_snrs"],
                   training_draws=sum(row["draws"] for row in history), epochs_completed=len(history),
                   fit_seconds=fit_seconds, setup_seconds_this_session=setup_seconds, stop_reason=stop_reason,
                   hardware=infer.hardware(device), manifest_sha256=helpers.lookup_sha256(manifest),
                   inference_inputs="model.pt and anonymous participant package only", external_data=False, pretrained=False,
                   notes="No training waveforms, template bank, teacher or saved predictions are used at inference. "
                         "All 13 SNRs are available in training after the two warm-up epochs. "
                         "The 0.996 goal has not been guaranteed. Full/audit scores are measured after training.")
    helpers.write_json(export / "training_summary.json", summary)
    (export / "method_summary.txt").write_text(
        "Track 2: compact complex-STFT neural classifier, trained from random initialization.\n"
        "Explicit two-sided complex STFT: 256 frequency bins x 66 time frames, 256-sample rectangular windows, "
        "non-overlapping blocks and right zero padding. Phase-sensitive learned projection plus a small power-STFT CNN.\n"
        "Gradient training: class-balanced source-pulse sampling, all 13 noise levels after warm-up, "
        "paired clean/noisy cross-entropy and feature consistency; raw/EMA checkpoint selection on validation only.\n"
        "Training cache contains training identities only and is excluded from the export. No validation pulse fits weights.\n"
        f"Full validation balanced accuracy: {report['mean_balanced_accuracy']:.8f}; separate audit: {audit_report['mean_balanced_accuracy']:.8f}.\n"
        "Training, hardware and SNR details: training_summary.json and training_config.json.\n"
        "Inference needs model.pt, radarml_track2_neural_infer.py and the blind package only.\n"
        "Limitations: validation covers known waveform configurations; blind accuracy, unseen-configuration "
        "generalization and official runtime eligibility are not guaranteed.\n"
        "Attribution: Ritchie, White and Hosford (2025), RadarML, https://doi.org/10.5522/04/30752767.v1; "
        "organizer GPL-3.0 data helpers; AI coding assistance: OpenAI Codex.\n"
        "API references: https://docs.pytorch.org/docs/stable/generated/torch.functional.stft.html and "
        "https://docs.pytorch.org/docs/stable/notes/amp_examples.html\n", encoding="utf-8")
    helpers.write_json(export / "complete.json", dict(model_sha256=infer.sha256(export / "model.pt")))
    print("EXPORTED MODEL:", export / "model.pt", flush=True)
    print("FULL VALIDATION:", report["mean_balanced_accuracy"], "| AUDIT:", audit_report["mean_balanced_accuracy"], flush=True)
    return report


def package(run):
    """Package inference artifacts and resume weights; NEVER package the training cache."""
    run = Path(run)
    model_zip = run.with_name(run.name + "_model.zip")
    with zipfile.ZipFile(model_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted((run / "export").iterdir()):
            if path.is_file():
                archive.write(path, path.name)
    resume_zip = run.with_name(run.name + "_resume.zip")
    with zipfile.ZipFile(resume_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ("resume.pt", "best.pt", "run_config.json", "history.json", "best_selection_validation.json"):
            if (run / name).exists():
                archive.write(run / name, name)
    return model_zip, resume_zip


## Training setup and representation

The input is converted to a **256-frequency-bin × 66-time-frame complex STFT**.
Windows are 256-sample rectangular blocks, hop 256, with right zero padding and no
centering. This explicit time–frequency transform preserves every input sample.
Both real and imaginary parts feed learned phase-sensitive neural features. A small
separable-convolution branch also learns modulation shapes from log STFT power.
No filename, waveform-configuration identity, pulse ID or requested-SNR label enters
the network. Per-pulse RMS, noise and clipping measurements come from the signal.

All **52,800 training source pulses** are eligible. The trainer caches their original
recorded waveforms, or their +30 dB variants if using expanded inputs: **3.55 GB** of
training-only int16 data. It then generates **independent noisy views at all 13 SNRs**
during training, following the supplied power windows and int16 clipping convention.
It does **not** fit only clean pulses. It also does not read every stored noisy variant
from the 57 GB expansion; the independent views are training-pulse augmentations.
This avoids repeatedly reading 57 GB and permits fresh noise on repeated visits.

Class-balanced sampling draws **105,600 examples per epoch**, with replacement.
Every source pulse is eligible; an epoch does not promise to visit every pulse.
After two warm-up epochs (+30 through −15 dB), all 13 SNRs are sampled, with twice
as much sampling weight at −20/−25/−30 dB to focus on the earlier model's weak region.
Paired clean/noisy training combines cross-entropy and feature consistency.

AdamW, mixed-precision training, gradient clipping, warm-up/cosine learning rates and
EMA weights are used. The selected export is **one network**, chosen between raw and
EMA checkpoints using validation. There is no inference ensemble or template teacher.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd()))
import radarml_track2_runtime as helpers
import radarml_track2_neural_infer as neural
import radarml_track2_neural_train as training
helpers = importlib.reload(helpers)
neural = importlib.reload(neural)
training = importlib.reload(training)
torch.set_num_threads(INFERENCE_CPU_THREADS)

CONFIG = copy.deepcopy(training.DEFAULT_CONFIG)
CONFIG.update(batch_size=TRAIN_BATCH_SIZE, workers=WORKERS, max_run_hours=TRAINING_BUDGET_HOURS)
# Make experiment changes before starting. Keep settings unchanged when resuming.
# CONFIG["epochs"] = 80
# CONFIG["model_kwargs"]["projection_rank"] = 64
# CONFIG["samples_per_epoch"] = 105600

if TRAIN_MODEL:
    DATA_ROOT = training.find_data_root(DATA_ROOT, DATA_FORMAT)
    manifest = helpers.find_manifest(DATA_ROOT, MANIFEST_PATH)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Development data:", DATA_ROOT)
    print("Published manifest:", manifest)
    print(json.dumps(neural.hardware(device), indent=2))
    print(json.dumps(CONFIG, indent=2))
    if device.type != "cuda":
        raise RuntimeError("Enable a Kaggle GPU before full training. CPU inference is supported after training.")
else:
    print("Inference-only: development data is not loaded.")


## Train, validate and export

The first four validation pulse indices per recording, across all 13 SNRs, select
raw/EMA checkpoints using **mean balanced accuracy per requested SNR**, with the
lowest-three-SNR tie-break. No validation pulse fits weights. The selected checkpoint
then receives full validation on **171,600 examples**, plus an audit on **137,280**
examples excluding all selection source pulses. Do not tune using the audit.

Checkpoint files are saved after each epoch. If interrupted during an epoch, rerunning
with the same configuration resumes from the previous saved epoch. The history shows
how many examples were actually drawn and the requested training SNR counts.
The time-budget check can stop a partial epoch; that partial epoch is recorded and
evaluated before export. Early stopping can end the run before 80 epochs.

A fresh run takes time to verify FSK upload identities and build its training cache.
Once caching is complete, it prints epoch/batch loss and validation progress.
The exported `model.pt` contains **neural weights and architecture settings only**.
No raw/averaged training waveform, cached example or old submission is exported.

In [ ]:
if TRAIN_MODEL:
    try:
        full_report = training.train(DATA_ROOT, manifest, RUN_DIR, CONFIG, kind=DATA_FORMAT)
    except KeyboardInterrupt:
        print("Interrupted. The preceding completed epoch is saved in resume.pt.")
        print("Save the notebook outputs before ending the session; resume requires best.pt and run_config.json too.")
        raise
    model_zip, resume_zip = training.package(RUN_DIR)
    MODEL_PATH = RUN_DIR / "export/model.pt"
    print("MODEL:", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1e6:.2f} MB)")
    print("MODEL + INFERENCE CODE:", model_zip)
    print("RESUME CHECKPOINTS (not for submission):", resume_zip)
    if REMOVE_TRAINING_CACHE_AFTER_EXPORT and (RUN_DIR / "training_cache").exists():
        shutil.rmtree(RUN_DIR / "training_cache")
        print("Training cache removed after export; it is not part of any submission artifact.")
    from IPython.display import FileLink, display
    display(FileLink(str(Path(model_zip).relative_to(WORK_ROOT))))
    display(FileLink(str(Path(resume_zip).relative_to(WORK_ROOT))))
    display(FileLink(str(MODEL_PATH.relative_to(WORK_ROOT))))
    display(FileLink(str((RUN_DIR / "export/radarml_track2_neural_infer.py").relative_to(WORK_ROOT))))


## Review measured accuracy

These are this neural network's **actual measured validation scores**, not the earlier
matched-filter model's scores. Reaching 0.996 cannot be assumed. If the result is
lower, preserve the reported score; never substitute the template bank's result.
Plots are saved outside the model file. Blind labels are not available to this notebook.

In [ ]:
if TRAIN_MODEL:
    export = RUN_DIR / "export"
    full = json.loads((export / "validation_report.json").read_text())
    audit = json.loads((export / "audit_validation.json").read_text())
    summary = json.loads((export / "training_summary.json").read_text())
    assert full["samples"] == 171600 and audit["samples"] == 137280
    print("NEURAL MODEL FULL VALIDATION:", full["mean_balanced_accuracy"])
    print("SEPARATE AUDIT:", audit["mean_balanced_accuracy"])
    print("Selected epoch:", full["selected_epoch"], "variant:", full["selected_variant"])
    print("Model parameters:", summary["parameters"], "model MB:", summary["model_bytes"] / 1e6)
    print("Requested SNR | balanced accuracy")
    for row in full["per_snr"]:
        print(f"{row['requested_snr_db']:+3d} | {row['balanced_accuracy']:.6f}")
    plots = RUN_DIR / "plots"
    plots.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(9, 4))
    for label, report in [("Full validation", full), ("Audit excluding selection", audit)]:
        ax.plot([r["requested_snr_db"] for r in report["per_snr"]],
                [r["balanced_accuracy"] for r in report["per_snr"]], "o-", label=label)
    ax.set(xlabel="Requested SNR (dB)", ylabel="Balanced accuracy", ylim=(0, 1.01))
    ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
    fig.savefig(plots / "validation_snr.png", dpi=160); plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, snr in zip(axes, (-20, -25, -30)):
        row = next(r for r in full["per_snr"] if r["requested_snr_db"] == snr)
        cm = np.array(row["confusion_matrix"], dtype=float)
        ax.imshow(cm / cm.sum(1, keepdims=True), vmin=0, vmax=1, cmap="Blues")
        ax.set(title=f"Requested {snr:+d} dB", xlabel="Predicted", ylabel="True")
        ax.set_xticks(range(7), neural.CLASSES, rotation=45, ha="right")
        ax.set_yticks(range(7), neural.CLASSES)
    fig.tight_layout(); fig.savefig(plots / "low_snr_confusions.png", dpi=160); plt.show()


## Standalone inference-script cell — generate submission and measure CPU latency

This is the same script that goes to the organizer. It reads **only `model.pt` and
the blind package**. The model includes its architecture settings, so no training
configuration, training cache, validation report, previous predictions, or optimizer
checkpoint is required. It emits seven class logits in one learned forward pass.

The default CPU run measures **1,000 individual signals**, then regenerates predictions
for all **171,600** samples. Single-signal latency includes shard access, int16 I/Q
conversion, the entire STFT, neural inference and class selection. It reports mean,
median, p95, p99 and maximum. Batched elapsed time per sample is reported separately
as throughput. One-time model loading is excluded and reported separately.
A 10 ms reference is shown for context; the organizer's exact timing protocol governs.

The output columns default to `sample_id,label`, matching your latest saved CSV.
Set `SUBMISSION_LABEL_COLUMN="predicted_label"` if the organizer requests that header.
No previous submission is opened to infer the column name or make predictions.

**Standalone command**, using the exported script and neural weights:

```bash
python radarml_track2_neural_infer.py --model model.pt \
  --participants /path/to/participants --output submission.csv \
  --device cpu --threads 4 --label-column label
```

In [ ]:
if GENERATE_SUBMISSION:
    if TRAIN_MODEL:
        model_path = RUN_DIR / "export/model.pt"
        script_path = RUN_DIR / "export/radarml_track2_neural_infer.py"
    else:
        model_path = Path(MODEL_PATH).expanduser().resolve()
        script_path = model_path.parent / "radarml_track2_neural_infer.py"
    if not script_path.is_file():
        raise FileNotFoundError("Keep the exported inference script beside model.pt")
    participants = neural.find_participants(BLIND_ROOT)
    command = [sys.executable, str(script_path), "--model", str(model_path),
               "--participants", str(participants), "--output", str(PREDICTIONS_PATH),
               "--device", INFERENCE_DEVICE, "--batch-size", str(INFERENCE_BATCH_SIZE),
               "--threads", str(INFERENCE_CPU_THREADS), "--workers", "0",
               "--benchmark-samples", str(BENCHMARK_SAMPLES),
               "--label-column", SUBMISSION_LABEL_COLUMN]
    subprocess.run(command, check=True)
    report_path = PREDICTIONS_PATH.with_name(PREDICTIONS_PATH.name + ".inference.json")
    print(json.loads(report_path.read_text()))
    from IPython.display import FileLink, display
    display(FileLink(str(PREDICTIONS_PATH.relative_to(WORK_ROOT))))
    display(FileLink(str(report_path.relative_to(WORK_ROOT))))
else:
    print("Submission generation skipped because MODE='train'.")


## Resume and report honestly

The `*_resume.zip` contains training weights/optimizer checkpoints, not cached source
pulses. Extract it as a private Kaggle dataset, set `RESUME_FROM`, retain the exact
configuration and Run All. The training-only cache is rebuilt from the supplied data.
Kaggle must retain your outputs for a later session; an abruptly killed session can
lose files that were never saved. If you need a new experiment, change `RUN_NAME`.

For final submission, use the **exported neural model and inference script**, not
`resume.pt`, the old `bank.pt`, the original CNN's script, or the training cache.
The model ZIP includes measured validation evidence and method/hardware details.

The implementation uses standard APIs documented by
[PyTorch STFT](https://docs.pytorch.org/docs/stable/generated/torch.functional.stft.html),
[automatic mixed precision](https://docs.pytorch.org/docs/stable/notes/amp_examples.html),
and [EMA model averaging](https://docs.pytorch.org/docs/stable/generated/torch.optim.swa_utils.AveragedModel.html).
Data attribution: Ritchie, White and Hosford (2025),
[RadarML](https://doi.org/10.5522/04/30752767.v1). Organizer helpers are GPL-3.0.
AI coding assistance: OpenAI Codex. No pretrained weights or external training data.